## Prepare data

In [1]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Get dataset path
import os
zip_path = '/content/drive/My Drive/DeepLearning/data.7z'
print(os.path.exists(zip_path))   # Should print True
print(os.path.getsize(zip_path))  # Check file size in bytes

True
11331643882


In [3]:
#Move the file to the current working directory

!cp "/content/drive/My Drive/DeepLearning/data.7z" "/content/data.7z"

In [4]:
!ls -lh /content/data.7z

# Test integrity before copying
!zip -T "/content/data.7z"

-rw------- 1 root root 11G Apr  8 11:39 /content/data.7z
	zip warning: missing end signature--probably not a zip file (did you
	zip warning: remember to use binary mode when you transferred it?)
	zip warning: (if you are trying to read a damaged archive try -F)

zip error: Zip file structure invalid (/content/data.7z)


In [5]:
print("Unzipping...")
#!unzip -q "/content/LG-SGNet.7z" -d "/content/dataset"
!7z x "/content/data.7z"
#-o"/content/dataset"
print("Done unzipping!")

# Verify contents
!ls /content

Unzipping...

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content                  1 file, 11331643882 bytes (11 GiB)

Extracting archive: /content/data.7z
--
Path = /content/data.7z
Type = 7z
Physical Size = 11331643882
Headers Size = 565003
Method = LZMA2:25
Solid = +
Blocks = 6

      0% 525 - data/nturgbd_raw/denoised_data/actors_info/S001C002P008R001A055.t                                                                              0% 976 - data/nturgbd_raw/denoised_data/actors_info/S002C002P003R001A060.t                                                                              0% 1292 - data/nturgbd_raw/denoised_data/actors_info/S002C003P010R002A059.tx                                                                                0% 1552 - data/nturgbd_raw/denoised_data/actors_info/S

## Dependency

In [6]:
%pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.8 MB/s eta 0:00:00:00:0100:01


In [7]:
!uv pip install -r requirements1.txt
!uv pip install -e torchlight

Using Python 3.12.13 environment at: /usr
Resolved 81 packages in 1.82s                                        
Prepared 15 packages in 1.30s                                            
Installed 15 packages in 14ms                               
 + fvcore==0.1.5.post20221221
 + iopath==0.1.10
 + jsonform==0.0.2
 + jsonsir==0.0.2
 + loguru==0.7.3
 + msgpack-numpy==0.4.8
 + multimethod==2.0.2
 + portalocker==3.2.0
 + python-easyconfig==0.1.7
 + resource==0.2.1
 + tensorboardx==2.6.5
 + tensorpack==0.11
 + thop==0.1.1.post2209072238
 + torchpack==0.3.1
 + yacs==0.1.8
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 562ms                                          
Prepared 1 package in 221ms                                              
Installed 1 package in 1msom file:///content/torchlight)    
 + torchlight==1.0 (from file:///content/torchlight)


## Train & Testing

In [8]:
!mkdir -p /content/data/ntu
!mv /content/data/nturgbd_raw/NTU60_CS.npz /content/data/ntu/NTU60_CS.npz
!mv /content/data/nturgbd_raw/NTU60_CV.npz /content/data/ntu/NTU60_CV.npz

In [9]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
Tesla T4


In [10]:
!nvidia-smi

Wed Apr  8 12:06:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!python main.py \
    --config config/nturgbd-cross-subject/default.yaml \
    --work-dir ./work_dir/ntu60/joint --model model.LG-SGNet.Model \
    --num-worker 2
    # --weights pretrained_model/...

<class 'model.LG-SGNet.Model'>
Model(
  (data_bn): BatchNorm1d(150, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (l1): LGSGNet_Block(
    (gcn1): LGGCN_unit(
      (convs): ModuleList(
        (0-2): 3 x LGGC(
          (conv1): Conv2d(3, 8, kernel_size=(1, 1), stride=(1, 1))
          (conv2): Conv2d(3, 8, kernel_size=(1, 1), stride=(1, 1))
          (conv3): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
          (conv4): Conv2d(8, 64, kernel_size=(1, 1), stride=(1, 1))
          (conv5): Conv2d(3, 8, kernel_size=(1, 1), stride=(1, 1))
          (conv6): Conv2d(1, 1, kernel_size=(1, 1), stride=(1, 1))
          (conv7): Conv2d(1, 1, kernel_size=(1, 1), stride=(1, 1))
          (tanh): Tanh()
        )
      )
      (down): Sequential(
        (0): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, 

In [ ]:
# Save work_dir
!mv "/content/work_dir" "/content/drive/My Drive/DeepLearning/"

In [ ]:
import time
from datetime import datetime

while True:
    print("Colab keep-alive:", datetime.now())
    time.sleep(120)  # wait 60 seconds

Colab keep-alive: 2026-03-11 22:24:18.267005


KeyboardInterrupt: 